# Improved competition pipeline — local evaluation

This notebook evaluates both subtasks using **only `train.xlsx`**.

- **Task 1A:** risk-level classification — weighted F1
- **Task 1B:** evidence extraction — Phrase F1
- **Task 2:** suicide-factor identification — Macro F1

The split is grouped by `anon_user_id`, so one author's posts cannot occur in both training and validation. This notebook intentionally does **not** predict `leaderboard.xlsx`.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedGroupKFold

from improved_competition_pipeline import (
    RISK, FACTORS, parse_factors, spans, make_vectors,
    best_thresholds, official_phrase_f1, evidence,
)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MultiLabelBinarizer

SEED = 42
N_FOLDS = 5
TRAIN_PATH = Path('train.xlsx')
assert TRAIN_PATH.exists(), f'Missing {TRAIN_PATH.resolve()}'



## Load and validate training data

In [3]:
train = pd.read_excel(TRAIN_PATH)
train['post'] = train['post'].fillna('').astype(str)
train['risk_level'] = train['suicide risk'].astype(str).str.strip().str.title()
assert set(train['risk_level']) <= set(RISK)

risk_y = np.array([RISK.index(x) for x in train['risk_level']])
mlb = MultiLabelBinarizer(classes=FACTORS)
factor_y = mlb.fit_transform(train['factors'].map(parse_factors))

print('Rows:', len(train))
print('Users:', train['anon_user_id'].nunique())
display(train['risk_level'].value_counts().reindex(RISK).to_frame('count'))
display(pd.DataFrame({'factor': FACTORS, 'count': factor_y.sum(axis=0)}).sort_values('count'))

Rows: 1635
Users: 153


,count
risk_level,
Indicator,611
Ideation,519
Behavior,391
Attempt,114


,factor,count
18,sexual orientation related issues,8
13,exposure to others' suicide,14
6,poor school performance,16
2,substance use,33
16,cognitive deficits,33
23,meaning in life,45
7,low socio-economic status,54
22,sense of responsibility,58
15,traumatic experience,64
1,physical health/characteristic,78


## Grouped out-of-fold training

Every row is predicted by a model that did not train on that row or any other post from the same author.

In [4]:
cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
risk_oof = np.zeros((len(train), len(RISK)), dtype=float)
factor_oof = np.zeros_like(factor_y, dtype=float)
fold_of_row = np.full(len(train), -1)

for fold, (fit_idx, val_idx) in enumerate(
    cv.split(train['post'], risk_y, train['anon_user_id']), start=1
):
    fit_users = set(train.iloc[fit_idx]['anon_user_id'])
    val_users = set(train.iloc[val_idx]['anon_user_id'])
    assert fit_users.isdisjoint(val_users)

    x_fit, x_val = make_vectors(train.iloc[fit_idx]['post'], train.iloc[val_idx]['post'])
    risk_model = LogisticRegression(
        C=4, max_iter=2500, class_weight='balanced', solver='lbfgs'
    )
    risk_model.fit(x_fit, risk_y[fit_idx])
    risk_oof[val_idx] = risk_model.predict_proba(x_val)

    for label_idx in range(len(FACTORS)):
        target = factor_y[fit_idx, label_idx]
        if target.min() == target.max():
            factor_oof[val_idx, label_idx] = target[0]
        else:
            factor_model = LogisticRegression(
                C=2, max_iter=1500, class_weight='balanced', solver='liblinear'
            )
            factor_model.fit(x_fit, target)
            factor_oof[val_idx, label_idx] = factor_model.predict_proba(x_val)[:, 1]

    fold_of_row[val_idx] = fold
    fold_f1 = f1_score(risk_y[val_idx], risk_oof[val_idx].argmax(axis=1), average='weighted')
    print(f'Fold {fold}: Task 1 risk weighted F1 = {fold_f1:.4f}')

assert (fold_of_row > 0).all()

Fold 1: Task 1 risk weighted F1 = 0.5948
Fold 2: Task 1 risk weighted F1 = 0.6376
Fold 3: Task 1 risk weighted F1 = 0.6353
Fold 4: Task 1 risk weighted F1 = 0.6258
Fold 5: Task 1 risk weighted F1 = 0.6756


## Task 1A — risk-level evaluation

In [5]:
risk_pred_id = risk_oof.argmax(axis=1)
risk_weighted_f1 = f1_score(risk_y, risk_pred_id, average='weighted')
print(f'Task 1A OOF weighted F1: {risk_weighted_f1:.4f}')
print(classification_report(risk_y, risk_pred_id, target_names=RISK, digits=4))

Task 1A OOF weighted F1: 0.6344
              precision    recall  f1-score   support

   Indicator     0.7126    0.7872    0.7481       611
    Ideation     0.6359    0.6763    0.6555       519
    Behavior     0.5404    0.4962    0.5173       391
     Attempt     0.5510    0.2368    0.3313       114

    accuracy                         0.6440      1635
   macro avg     0.6100    0.5491    0.5630      1635
weighted avg     0.6358    0.6440    0.6344      1635



## Task 1B — evidence Phrase F1

Evidence is extracted verbatim from each post using its out-of-fold predicted risk level. The scorer implements case-insensitive containment, one-to-one matching, and the 3× token-length limit.

In [6]:
gold_evidence = train['evidence for suicide risk level'].map(spans).tolist()
predicted_risk = [RISK[i] for i in risk_pred_id]
predicted_evidence = [evidence(post, risk) for post, risk in zip(train['post'], predicted_risk)]
phrase_f1 = official_phrase_f1(gold_evidence, predicted_evidence)
print(f'Task 1B OOF Phrase F1: {phrase_f1:.4f}')
print(f'Task 1 combined score (0.4*risk + 0.3*evidence): {0.4*risk_weighted_f1 + 0.3*phrase_f1:.4f}')

evidence_examples = train[['row_id', 'post', 'risk_level']].copy()
evidence_examples['predicted_risk'] = predicted_risk
evidence_examples['gold_evidence'] = gold_evidence
evidence_examples['predicted_evidence'] = predicted_evidence
display(evidence_examples.sample(10, random_state=SEED))

Task 1B OOF Phrase F1: 0.1713
Task 1 combined score (0.4*risk + 0.3*evidence): 0.3051


,row_id,post,risk_level,predicted_risk,gold_evidence,predicted_evidence
937,P01439,Had a weird dream where I became paralyzed and...,Indicator,Indicator,[Main phrases that lead to this assessment: none],[]
1097,P01701,if you actually cared then why wont you take m...,Indicator,Indicator,[Main phrases that lead to this assessment: none],[]
813,P01264,someone who can undestand ocd i need to talk t...,Indicator,Indicator,[],[]
607,P00932,I’m a creep and I need to die I’m a 21m and I ...,Ideation,Ideation,"[need to die, end it]",[]
1304,P01949,Suicidal notes You’ll learn to live without me...,Behavior,Behavior,[Main phrases that lead to this assessment: Su...,[]
1074,P01643,On my 19th birthday I said I hope this is my l...,Behavior,Behavior,[Main phrases that lead to this assessment: I ...,[]
839,P01306,I am so tired I am tired of having to put on a...,Indicator,Indicator,[Main phrases that lead to this assessment: none],[]
809,P01260,i think i need someone to talk but i dont know...,Indicator,Indicator,[],[]
1237,P01874,if only failed attempts weren't a thing i can'...,Attempt,Attempt,[Main phrases that lead to this assessment: fa...,[]
271,P00394,I'm very lonely. I feel like there's something...,Behavior,Ideation,[almost hurt myself],[]


## Task 2 — factor evaluation

A separate threshold is learned for every factor from out-of-fold probabilities. This directly optimizes the per-label F1 values used by Macro F1.

In [7]:
factor_thresholds = best_thresholds(factor_y, factor_oof)
factor_pred = factor_oof >= factor_thresholds
factor_macro_f1 = f1_score(factor_y, factor_pred, average='macro', zero_division=0)
print(f'Task 2 OOF Macro F1: {factor_macro_f1:.4f}')

per_factor = []
for j, label in enumerate(FACTORS):
    per_factor.append({
        'factor': label,
        'support': int(factor_y[:, j].sum()),
        'threshold': factor_thresholds[j],
        'f1': f1_score(factor_y[:, j], factor_pred[:, j], zero_division=0),
    })
display(pd.DataFrame(per_factor).sort_values('f1'))

Task 2 OOF Macro F1: 0.4269


,factor,support,threshold,f1
18,sexual orientation related issues,8,0.050,0.000000
13,exposure to others' suicide,14,0.100,0.135593
16,cognitive deficits,33,0.200,0.169811
2,substance use,33,0.175,0.185185
6,poor school performance,16,0.125,0.190476
23,meaning in life,45,0.300,0.204082
1,physical health/characteristic,78,0.225,0.295181
22,sense of responsibility,58,0.350,0.299320
7,low socio-economic status,54,0.275,0.347107
19,social support,112,0.275,0.375000


## Local composite estimate

The official total is `0.4 × Task 1 risk F1 + 0.3 × evidence Phrase F1 + 0.3 × Task 2 Macro F1`.

In [8]:
local_composite = 0.4 * risk_weighted_f1 + 0.3 * phrase_f1 + 0.3 * factor_macro_f1
print(f'Local OOF composite estimate: {local_composite:.4f}')

Local OOF composite estimate: 0.4332


## Leaderboard prediction — intentionally postponed

No leaderboard file is generated by this notebook. After the local results are reviewed, the final stage should retrain every model on all rows in `train.xlsx`, reuse the learned factor thresholds, predict `leaderboard.xlsx`, validate the submission fields, and save `YourTeamName.csv`.